In [2]:
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from gensim.models import Word2Vec
from tqdm.auto import tqdm

In [3]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
import numpy as np

# 1. Загружаем нужные категории данных
selected_classes = [
    'alt.atheism',
    'comp.graphics',
    'sci.space',
    'talk.politics.mideast'
]

newsgroups = fetch_20newsgroups(
    subset='all',
    categories=selected_classes,
    remove=('headers', 'footers', 'quotes')
)

# 2. Разделяем на обучающую и тестовую выборки
texts_train, texts_test, y_train, y_test = train_test_split(
    newsgroups.data, newsgroups.target, test_size=0.2, random_state=42
)

print(f"Data loaded. Train size: {len(texts_train)}, Test size: {len(texts_test)}")

Data loaded. Train size: 2959, Test size: 740


In [ ]:
# 1. BERT Embeddings
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_name = 'distilbert-base-uncased' # Быстрее обычного BERT
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name).to(device)

def get_bert_embeddings(texts):
    embeddings = []
    for text in tqdm(texts, desc="BERT Embeddings"):
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
        with torch.no_grad():
            outputs = bert_model(**inputs)
            cls_emb = outputs.last_hidden_state[0, 0, :].cpu().numpy()
            embeddings.append(cls_emb)
    return np.array(embeddings)

X_train_bert = get_bert_embeddings(texts_train)
X_test_bert = get_bert_embeddings(texts_test)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT Embeddings:   0%|          | 0/2959 [00:00<?, ?it/s]

BERT Embeddings:   0%|          | 0/740 [00:00<?, ?it/s]

In [ ]:
import gensim.downloader as api
from gensim.utils import simple_preprocess

print("Loading pre-trained GloVe model...")

w2v_model = api.load("glove-wiki-gigaword-100")

def get_w2v_avg(texts, model):
    vectors = []
    for text in texts:
        tokens = simple_preprocess(text)
        
        valid_vectors = [model[word] for word in tokens if word in model]
        if valid_vectors:
            vectors.append(np.mean(valid_vectors, axis=0))
        else:
            vectors.append(np.zeros(model.vector_size))
    return np.array(vectors)

print("Extracting pre-trained Word2Vec embeddings...")
X_train_w2v = get_w2v_avg(texts_train, w2v_model)
X_test_w2v = get_w2v_avg(texts_test, w2v_model)

Loading pre-trained GloVe model...
Extracting pre-trained Word2Vec embeddings...


In [ ]:
def evaluate_embeddings(X_train, X_test, name):
    # Классификатор 1: Логистическая регрессия
    lr = LogisticRegression(max_iter=1000)
    lr.fit(X_train, y_train)
    acc_lr = accuracy_score(y_test, lr.predict(X_test))

    # Классификатор 2: Случайный лес
    rf = RandomForestClassifier(n_estimators=100)
    rf.fit(X_train, y_train)
    acc_rf = accuracy_score(y_test, rf.predict(X_test))

    print(f"Results for {name}:")
    print(f"  Logistic Regression: {acc_lr:.4f}")
    print(f"  Random Forest:       {acc_rf:.4f}\n")

evaluate_embeddings(X_train_bert, X_test_bert, "BERT (CLS Token)")
evaluate_embeddings(X_train_w2v, X_test_w2v, "Word2Vec (Average)")

Results for BERT (CLS Token):
  Logistic Regression: 0.8649
  Random Forest:       0.8405

Results for Word2Vec (Average):
  Logistic Regression: 0.8365
  Random Forest:       0.8014

